# 01 — Validación de datos

Revisa calidad antes de calcular nada: nulos, partidos no maestrados, IDs huérfanos, temas vacíos. **Sin esto los índices están sesgados.**


In [ ]:
# Cargar datos: cambia URL o XLSX segun corresponda
from pathlib import Path
import pandas as pd, numpy as np
from agendapp.io import fetch_endpoint, load_xlsx_municipio
from agendapp.transform import matriz_concejal_tema, binarizar, perfil_partido, filtrar_min_instrumentos
from agendapp.indices import shannon_norm, cv_shannon, jaccard_pairwise_mean, party_correlation
from agendapp import viz

# Opcion A: endpoint Apps Script (descomentar y poner URL real)
# URL = "https://script.google.com/macros/s/AKfycb.../exec"
# data = fetch_endpoint(URL)
# df_inst = pd.DataFrame(data["instrumentos"])

# Opcion B: xlsx local (piloto Guarne)
XLSX = Path("../..") / "Guarne_DILIGENCIADO.xlsx"
d = load_xlsx_municipio(XLSX)
df_inst = d["instrumentos"]
df_inst["Partido / Movimiento"] = df_inst["Partido / Movimiento"].astype(str).str.strip().str.upper()
df_inst.shape


## Conteo de filas por estado de inclusión


In [ ]:
df_inst['Incluir en analisis'].value_counts(dropna=False)


## Nulos en columnas clave


In [ ]:
cols_clave = ['Identificador','Anio','Rol','ID_Concejal','Partido / Movimiento','Sector','Tematica','Incluir en analisis']
df_inst[cols_clave].isna().mean().sort_values(ascending=False)


## Filas Incluir=Si sin Tematica


In [ ]:
mask = df_inst['Incluir en analisis'].astype(str).str.lower().eq('si') & df_inst['Tematica'].isna()
df_inst[mask][['Identificador','Titulo','Rol','ID_Concejal']].head(20)


## Partidos presentes (revisar si hay duplicados por tildes/mayúsculas)


In [ ]:
df_inst['Partido / Movimiento'].value_counts().head(30)


## Conteo de instrumentos por concejal (detecta huérfanos y bajo volumen)


In [ ]:
conteo = df_inst[df_inst['Rol'].isin(['Proponente', 'Ponente'])]['ID_Concejal'].value_counts()
print('Concejales con < 3 instrumentos como autor (Proponente/Ponente):')
conteo[conteo < 3]
